In [ ]:
# =========================================
# 1. Import Libraries
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
import os

In [ ]:
# =========================================
# 2. Load Dataset
# =========================================

path = r"D:\Neural Networks\DKHousingPricesSample100k.csv"
df = pd.read_csv(path)


In [ ]:
features = [
    "house_type",
    "sales_type",
    "year_build",
    "%_change_between_offer_and_purchase",
    "no_rooms",
    "sqm",
    "city",
    "area",
    "region",
    "nom_interest_rate%",
    "dk_ann_infl_rate%",
    "yield_on_mortgage_credit_bonds%"
]

target = "purchase_price"

df = df.dropna(subset=features + [target])

X = df[features]
y = df[target].values.reshape(-1, 1)

In [ ]:
# =========================================
# 4. Preprocessing 
# =========================================
numeric_features = [
    "year_build",
    "%_change_between_offer_and_purchase",
    "no_rooms",
    "sqm",
    "nom_interest_rate%",
    "dk_ann_infl_rate%",
    "yield_on_mortgage_credit_bonds%"
]

categorical_features = [
    "house_type",
    "sales_type",
    "city",
    "area",
    "region"
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_processed = preprocessor.fit_transform(X)

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y)

In [ ]:
# =========================================
# 5. Train/Test Split
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y_scaled,
    test_size=0.2,
    random_state=42
)


In [ ]:
# =========================================
# 6. Convert to PyTorch Tensors
# =========================================

X_train_t = torch.FloatTensor(X_train)

y_train_t = torch.FloatTensor(y_train)

X_test_t = torch.FloatTensor(X_test)

y_test_t = torch.FloatTensor(y_test)

print(X_train_t.shape)
print(y_train_t.shape)

In [ ]:
# =========================================
# 7. Build RELU and sigmoid Neural Network Model
# =========================================
class ReLUModel(nn.Module):
    def __init__(self, input_dim):
        super(ReLUModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )
    def forward(self, x): return self.net(x)


class SigmoidModel(nn.Module):
    def __init__(self, input_dim):
        super(SigmoidModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.Sigmoid(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.Sigmoid(),
            nn.Linear(256, 1)
        )
    def forward(self, x): return self.net(x)

In [ ]:
# =========================================
# 9. Training Function
# =========================================
def train_model(model_class, epochs=300):

    model = model_class(X_train.shape[1])

    criterion = nn.MSELoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-5
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=15
    )

    save_dir = r"D:\Neural Networks"

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    save_path = os.path.join(
        save_dir,
        f'best_{model_class.__name__}.pth'
    )

    history = {
        'train_loss': [],
        'val_loss': [],
        'r2': []
    }

    best_loss = float('inf')

    print(f"{'Epoch':<10} | {'Train Loss':<12} | {'Val MSE':<12} | {'R2 Score':<10}")
    print("-" * 55)

    for epoch in range(1, epochs + 1):

    
        # Training
        model.train()

        optimizer.zero_grad()

        output = model(X_train_t)

        loss = criterion(output, y_train_t)

        loss.backward()

        optimizer.step()

      
        # Validation
        model.eval()

        with torch.no_grad():

            val_output = model(X_test_t)

            v_loss = criterion(val_output, y_test_t)

            r2 = r2_score(
                y_test_t.numpy(),
                val_output.numpy()
            )

        scheduler.step(v_loss)

        # ======================
        # Save Best Model
        # ======================
        if v_loss < best_loss:

            best_loss = v_loss

            torch.save(
                model.state_dict(),
                save_path
            )

        history['train_loss'].append(loss.item())
        history['val_loss'].append(v_loss.item())
        history['r2'].append(r2)

        if epoch % 50 == 0:

            print(
                f"{epoch:<10} | "
                f"{loss.item():<12.4f} | "
                f"{v_loss.item():<12.4f} | "
                f"{r2:<10.4f}"
            )

    print(f"\nBest model saved at: {save_path}")

    return model, history

In [ ]:
# =========================================
# 11. Final Evaluation
# =========================================
relu_model, relu_history = train_model(
    ReLUModel,
    epochs=300
)

relu_mse = relu_history['val_loss'][-1]
relu_r2 = relu_history['r2'][-1]

print("\n" + "="*50)
print("ReLU Model Evaluation")
print("="*50)

print(f"Mean Squared Error (MSE): {relu_mse:.6f}")
print(f"R2 Score: {relu_r2:.4f}")
print(f"Final Train Loss: {relu_history['train_loss'][-1]:.6f}")

print("="*50)

# Train Sigmoid Model
sigmoid_model, sigmoid_history = train_model(
    SigmoidModel,
    epochs=300
)

sigmoid_mse = sigmoid_history['val_loss'][-1]
sigmoid_r2 = sigmoid_history['r2'][-1]

print("\n" + "="*50)
print("Sigmoid Model Evaluation")
print("="*50)

print(f"Mean Squared Error (MSE): {sigmoid_mse:.6f}")
print(f"R2 Score: {sigmoid_r2:.4f}")
print(f"Final Train Loss: {sigmoid_history['train_loss'][-1]:.6f}")

print("="*50)

: 

In [ ]:
# =========================================
# 12. Visualization
# =========================================
plt.figure(figsize=(16, 6))

# =========================================
# Loss Curves
# =========================================
plt.subplot(1, 2, 1)

# ReLU
plt.plot(
    relu_history['train_loss'],
    label='ReLU Train Loss'
)

plt.plot(
    relu_history['val_loss'],
    label='ReLU Validation Loss'
)

# Sigmoid
plt.plot(
    sigmoid_history['train_loss'],
    label='Sigmoid Train Loss'
)

plt.plot(
    sigmoid_history['val_loss'],
    label='Sigmoid Validation Loss'
)

plt.title('Training & Validation Loss')

plt.xlabel('Epochs')

plt.ylabel('Loss')

plt.legend()

plt.subplot(1, 2, 2)

# ReLU
plt.plot(
    relu_history['r2'],
    label='ReLU R2 Score'
)

# Sigmoid
plt.plot(
    sigmoid_history['r2'],
    label='Sigmoid R2 Score'
)

plt.axhline(
    y=0.6,
    color='red',
    linestyle='--',
    label='Target R2 = 0.6'
)

plt.title('R2 Score Comparison')

plt.xlabel('Epochs')

plt.ylabel('R2 Score')

plt.legend()

plt.tight_layout()

plt.show()